# 03 — Preprocess v1 (no training)

Paper order: pick frames that can show a leaf → resize **256×256** → count classes. Augment stays train-only.

v1 writes the **eight** `all` classes. *Thalassia hemprichii* is counted, not written.

This notebook calls `ml/src/export_processed.py`. On this laptop the dataset is `assets/species/Seagrass`. In Colab, mount Drive first and pass that root.

Do **not** treat the local dump as the Drive training root (*H. pinifolia* is smaller here).

The summary is appended to `ml/artifacts/run_log.md` with a timestamp. The latest copy remains `ml/artifacts/preprocess_v1.md`.

In [ ]:
from pathlib import Path
import sys

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False

def repo_root() -> Path:
    seen: set[Path] = set()
    for seed in [Path.cwd().resolve()]:
        for parent in [seed, *seed.parents]:
            for candidate in (parent, parent / "marinesee"):
                if candidate in seen:
                    continue
                seen.add(candidate)
                if (candidate / "ml" / "src" / "dataset_contract.py").is_file():
                    return candidate
    raise FileNotFoundError(
        "ml/src not found. Open this notebook from the MarineSee repo, or clone it in Colab."
    )

IN_COLAB = running_in_colab()
print("runtime", "colab" if IN_COLAB else "local machine")

if not IN_COLAB:
    print("Colab Drive mount skipped. Preprocess reads photos on this machine.")
else:
    mounted = Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists()
    if not mounted:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
    print("drive ok")

REPO = repo_root()
sys.path.insert(0, str(REPO / "ml" / "src"))
print("repo", REPO)

In [ ]:
from export_processed import run

if IN_COLAB:
    root = next(
        (
            cand
            for cand in (
                Path("/content/drive/MyDrive/Seagrass"),
                Path("/content/drive/My Drive/Seagrass"),
            )
            if cand.is_dir()
        ),
        None,
    )
    if root is None:
        raise FileNotFoundError("Seagrass dataset not found under My Drive")
    dest = Path("/content/drive/MyDrive/Seagrass-ml-artifacts/processed/v1")
else:
    root = REPO / "assets" / "species" / "Seagrass"
    if not root.is_dir():
        raise FileNotFoundError(root)
    dest = REPO / "data" / "processed" / "v1"

print("DATA_ROOT", root)
print("source", "google drive" if IN_COLAB else "this machine")
summary = run(
    root=root,
    dest=dest,
    report_json=REPO / "ml" / "artifacts" / "preprocess_v1.json",
    report_md=REPO / "ml" / "artifacts" / "preprocess_v1.md",
    write_images=True,
)
print(summary["kept"], "kept of", summary["listed"])
print(summary["by_slug_kept"])